# audio

> Audio file serving route handlers for alignment playback

In [ ]:
#| default_exp routes.core.audio

In [ ]:
#| export
from pathlib import Path
from typing import Tuple, Dict, Callable

from cjm_fasthtml_app_core.core.routing import APIRouter
from starlette.responses import FileResponse, Response

from cjm_fasthtml_interactions.core.state_store import get_session_id

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

DEBUG_AUDIO = False

## Handlers

In [ ]:
#| export
def _handle_audio_src(
    workflow:StructureDecompWorkflow,  # The workflow instance
    sess,  # FastHTML session object
    path:str=None,  # Audio file path (from query parameter)
):  # Audio file response or 404
    """Serve an audio file by path for Web Audio API playback."""
    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] path param: {path}")

    # Use provided path, or fall back to media_path from state
    if not path:
        session_id = get_session_id(sess)
        state = workflow.state_store.get_state(workflow.config.workflow_id, session_id)
        align_state = state.get("step_states", {}).get("alignment", {})
        path = align_state.get("media_path")

    if DEBUG_AUDIO:
        print(f"[AUDIO_SRC] resolved path: {path}")

    if not path or not Path(path).is_file():
        if DEBUG_AUDIO:
            print(f"[AUDIO_SRC] Returning 404 - path missing or file not found")
        return Response(status_code=404)

    return FileResponse(str(path), media_type="audio/mpeg")

## Router

In [ ]:
#| export
def init_audio_router(
    workflow: StructureDecompWorkflow,  # The workflow instance
    prefix: str,  # Route prefix (e.g., "/workflow/core/audio")
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, route_dict)
    """Initialize audio serving routes."""
    router = APIRouter(prefix=prefix)

    @router
    def audio_src(request, sess, path:str=None):
        """Serve an audio file by path for Web Audio API playback."""
        return _handle_audio_src(workflow, sess, path=path)

    routes = {
        "audio_src": audio_src,
    }

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()